# DeepCFD Training

In [3]:
import os, time, copy, pickle, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from src.models import build_model, count_params, PhysicsInformedLoss

In [4]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
with open("dataset/dataX.pkl", "rb") as f: X = pickle.load(f)
with open("dataset/dataY.pkl", "rb") as f: Y = pickle.load(f)
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)
idx = torch.randperm(len(X), generator=torch.Generator().manual_seed(42))
X, Y = X[idx], Y[idx]
n = len(X); n1, n2 = int(0.7*n), int(0.85*n)
train_ds = TensorDataset(X[:n1], Y[:n1])
val_ds   = TensorDataset(X[n1:n2], Y[n1:n2])
test_ds  = TensorDataset(X[n2:], Y[n2:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)
channel_weights = torch.sqrt((Y**2).mean(dim=(0,2,3)))
torch.save({"test_idx": idx[n2:], "val_idx": idx[n1:n2], "channel_weights": channel_weights}, "checkpoints/data_splits.pt")

In [6]:
def masked_mse(y_hat, y, weights):
    err = (y_hat - y)**2 
    loss = 0
    for i in range(3):
        loss += err[:,i].mean() / (weights[i]**2 + 1e-6)
    return loss / 3.0
def physics_ramp(epoch, max_epoch):
    return min(1.0, epoch / (max_epoch * 0.5))
def train_one_epoch(model, loader, opt, crit, phys_crit, lb_phys, device):
    model.train(); total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        y_hat = model(x)
        loss = crit(y_hat, y)
        if phys_crit:
            p_losses = phys_crit(y_hat, x)
            loss += lb_phys * p_losses["total_physics"]
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    return total / len(loader.dataset)
def evaluate(model, loader, crit, device):
    model.eval(); total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            total += crit(y_hat, y).item() * x.size(0)
    return total / len(loader.dataset)
def run_experiment(name, model, train_loader, val_loader, weights, use_physics=False, epochs=300):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=20)
    crit = lambda yh, y: masked_mse(yh, y, weights)
    phys_crit = PhysicsInformedLoss().to(device) if use_physics else None
    best_v = float("inf"); best_m = None; history = []
    for e in range(epochs):
        lb = 0.01 * physics_ramp(e, epochs) if use_physics else 0
        tl = train_one_epoch(model, train_loader, opt, crit, phys_crit, lb, device)
        vl = evaluate(model, val_loader, crit, device)
        sched.step(vl)
        history.append((tl, vl))
        if vl < best_v:
            best_v = vl; best_m = copy.deepcopy(model.state_dict())
    return best_m, best_v, history

In [7]:
CONFIG = {"filters": (8,16,32,32), "input_hw": (172,79), "epochs": 300}
results = {}
exps = [
("base_data", "base", False), ("attn_data", "attn", False), ("fno_data", "fno", False), ("trans_data", "trans", False),
("attn_phys", "attn", True), ("fno_phys", "fno", True), ("trans_phys", "trans", True)
]
for key, b_type, phys in exps:
    print(f"Running {key}...")
    m = build_model(b_type, filters=CONFIG["filters"], input_hw=CONFIG["input_hw"])
    sd, bv, hist = run_experiment(key, m, train_loader, val_loader, channel_weights, use_physics=phys, epochs=CONFIG["epochs"])
    ckpt = {"key": key, "state_dict": sd, "best_val_loss": bv, "builder": b_type, "use_physics": phys, "filters": CONFIG["filters"], "input_hw": CONFIG["input_hw"]}
    torch.save(ckpt, f"checkpoints/model_{key}.pt")
    results[key] = hist
with open("checkpoints/training_history.pkl", "wb") as f: pickle.dump(results, f)

Running base_data...
Running attn_data...
Running fno_data...
Running trans_data...
Running attn_phys...
Running fno_phys...
Running trans_phys...


Training completed